# Installing required packages

In [1]:
pip install beautifulsoup4 Requests pandas langdetect

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# Importing required libraries

In [1]:
from bs4 import BeautifulSoup
from pathlib import Path
import requests, csv, json, time
import pandas as pd
from langdetect import detect, DetectorFactory
import re

In [2]:
main_dir = Path.cwd()

# Validate if valid sentences

In [3]:
DetectorFactory.seed = 42

def is_english(text, threshold=0.5):

    if not isinstance(text, str):
        return False
    
    if not text or len(text.strip()) == 0:
        return False

    try:
        lang = detect(text)
        return lang == 'en'
    except:
        return is_english_chars(text, threshold)

def is_english_chars(text, threshold=0.5):

    if not text or len(text.strip()) == 0:
        return False
    
    english_chars = re.findall(r'[a-zA-Z0-9\s\.\,\!\?\'\"]', text)
    ratio = len(english_chars) / len(text)
    return ratio >= threshold

In [4]:
def insert_into_csv(platform, game_name, text):
    output_dir = main_dir / "text_data" / platform

    filename = "".join(c for c in game_name if c.isalnum() or c in (' ', '-', '_')).strip().replace(' ', '_')
    output_path = output_dir / f"{filename}.csv"

    if not is_english(text):
        return

    group_df = pd.DataFrame({"text": [text]})

    if output_path.exists():
        existing_df = pd.read_csv(output_path, encoding='utf-8')

        combined_df = pd.concat([existing_df, group_df], ignore_index=True)
        combined_df = combined_df.drop_duplicates(subset=['text'])

        combined_df[["text"]].to_csv(output_path, index=False, encoding='utf-8')
    else:
        group_df[["text"]].to_csv(output_path, index=False, encoding='utf-8')

# Initial values for Steam Data

In [4]:
FORUM_LIST_PAGE_SIZE_STEAM = 10
COMMENT_LIST_PAGE_SIZE_STEAM = 10

# Steam Web Scraping Process

In [9]:
with open(main_dir / "games_list" / "steam_urls.json", "r", encoding="utf-8") as f:
    game_urls = json.load(f)

print(game_urls)

{'Counter-Strike 2': {'url': 'https://steamcommunity.com/app/730/discussions/0/', 'scrap': False}, 'Helldivers 2': {'url': 'https://steamcommunity.com/app/553850/discussions/0/', 'scrap': False}, 'Dota 2': {'url': 'https://steamcommunity.com/app/570/discussions/0/', 'scrap': False}, 'Team Fortress 2': {'url': 'https://steamcommunity.com/app/440/discussions/0/', 'scrap': False}, 'Terraria': {'url': 'https://steamcommunity.com/app/105600/discussions/0/', 'scrap': False}, "Tom Clancy's Rainbow Six Siege": {'url': 'https://steamcommunity.com/app/359550/discussions/0/', 'scrap': False}, 'Marvel Rivals': {'url': 'https://steamcommunity.com/app/2767030/discussions/0/', 'scrap': True}, 'Dead by Daylight': {'url': 'https://steamcommunity.com/app/381210/discussions/0/', 'scrap': True}, 'Rocket League': {'url': 'https://steamcommunity.com/app/252950/discussions/0/', 'scrap': True}, 'Apex Legends': {'url': 'https://steamcommunity.com/app/1172470/discussions/0/', 'scrap': True}}


In [10]:
for game, url in game_urls.items():

    if url['scrap']:

        print("Scrapping Text for game: " + game + "...")

        for i in range(FORUM_LIST_PAGE_SIZE_STEAM):

            params = {
                    "fp" : (i + 1)
                }

            time.sleep(1)
            data = requests.get(url['url'], params=params)
            
            parent_html = data.text

            soup = BeautifulSoup(parent_html, 'lxml')

            forum_discussions = soup.find('div', class_='forum_area').find_all('a', class_='forum_topic_overlay')

            for discussion in forum_discussions:

                for j in range(COMMENT_LIST_PAGE_SIZE_STEAM):
                    child_params = {
                                    "ctp" : (j + 1)
                                }

                    time.sleep(1)
                    new_resp = requests.get(discussion['href'], params=child_params)

                    new_data = new_resp.text

                    child_soup = BeautifulSoup(new_data, 'lxml')

                    comments = child_soup.find_all('div', class_='commentthread_comment_text')

                    for comment in comments:

                        for child in comment.find_all():
                            child.extract()

                        text = comment.get_text(strip=True)

                        insert_into_csv("Steam", game, text)

    print('Completed\n')

Completed

Completed

Completed

Completed

Completed

Completed

Scrapping Text for game: Marvel Rivals...
Completed

Scrapping Text for game: Dead by Daylight...
Completed

Scrapping Text for game: Rocket League...
Completed

Scrapping Text for game: Apex Legends...
Completed



# Initial values for Reddit Data

In [5]:
SUBREDDIT_LIST_SIZE = 100
COMMENT_LIST_SIZE_REDDIT = 3

# Reddit Web Scraping Process

In [6]:
with open(main_dir / "games_list" / "reddit_urls.json", "r", encoding="utf-8") as f:
    game_urls = json.load(f)

print(game_urls)

{'League of Legends': {'url': 'https://www.reddit.com/r/leagueoflegends/new.rss', 'scrap': True}, 'Minecraft': {'url': 'https://www.reddit.com/r/Minecraft/new.rss', 'scrap': True}, 'Fortnite': {'url': 'https://www.reddit.com/r/Fortnite/new.rss', 'scrap': True}, 'World of Warcraft': {'url': 'https://www.reddit.com/r/wow/new.rss', 'scrap': True}, 'Heroes of the Storm': {'url': 'https://www.reddit.com/r/hots/new.rss', 'scrap': True}}


In [7]:
headers = {'User-agent': 'RedditScraper/0.1'}

params = {
    "limit": SUBREDDIT_LIST_SIZE
}

In [8]:
for game, url in game_urls.items():

    if url['scrap']:

        print("Scrapping Text for game: " + game + "...")

        data = requests.get(url['url'], headers=headers, params=params)

        soup = BeautifulSoup(data.text, 'xml')

        items = soup.find_all('entry')

        # print(items)

        for item in items:
            item_url = item.find('link')['href'][:-1] + '.rss'
            time.sleep(120)
            item_data = requests.get(item_url, headers=headers)
            item_soup = BeautifulSoup(item_data.text, 'xml')
            
            comments = item_soup.find_all('entry')

            # print(item_soup.find_all('entry'))

            for comment in comments:
                #print(comment.find('content').string)

                try:
                    html_elem = BeautifulSoup(comment.find('content').string.
                                                replace('<!-- SC_OFF -->', '').
                                                replace('<!-- SC_ON -->', ''), "lxml")
                
                    text = html_elem.find('div', class_='md').get_text(strip=True)
                    insert_into_csv("Reddit", game, text)
                except Exception:
                    pass
                # print(text)
        
        time.sleep(120)
            


    print('Completed\n')

Scrapping Text for game: League of Legends...
Completed

Scrapping Text for game: Minecraft...
Completed

Scrapping Text for game: Fortnite...
Completed

Scrapping Text for game: World of Warcraft...
Completed

Scrapping Text for game: Heroes of the Storm...
Completed

